# Train Ball Detector

This notebook trains a one-class YOLO detector from datasets exported by `tools/export_yolo/export_ball_dataset.py`.

Expected input after export:

```txt
data/exports/ball_yolo/
  dataset.yaml
  images/train/*.jpg
  images/val/*.jpg
  labels/train/*.txt
  labels/val/*.txt
```

## 1. Runtime Check

In Colab, choose `Runtime > Change runtime type > GPU` before training.

In [ ]:
import sys
import platform

print("python", sys.version)
print("platform", platform.platform())

try:
    import torch
    print("torch", torch.__version__)
    print("cuda available", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("gpu", torch.cuda.get_device_name(0))
except ImportError:
    print("torch not installed yet")

## 2. Install Training Dependencies

In [ ]:
%pip install -q ultralytics

## 3. Provide Dataset

Option A: upload a zipped exported dataset named `ball_yolo.zip` through the Colab file browser.

Option B: mount Google Drive and point `DATASET_YAML` at the exported `dataset.yaml`.

In [ ]:
from pathlib import Path
import zipfile

zip_path = Path("ball_yolo.zip")
extract_dir = Path("/content/ball_yolo")
if zip_path.exists():
    with zipfile.ZipFile(zip_path) as archive:
        archive.extractall(extract_dir)

matches = sorted(extract_dir.rglob("dataset.yaml"))
DATASET_YAML = str(matches[0]) if matches else str(extract_dir / "dataset.yaml")

print("dataset yaml:", DATASET_YAML)
print("exists:", Path(DATASET_YAML).exists())

## 4. Train Baseline

Start with the nano model. For a tiny fast ball, `imgsz=960` or `1280` is usually more useful than `640`, but it uses more memory.

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo26n.pt")
results = model.train(
    data=DATASET_YAML,
    epochs=100,
    imgsz=960,
    batch=-1,
    patience=25,
    project="/content/runs/ar_ping_pong",
    name="ball_yolo26n_img960",
)

## 5. Validate And Inspect Predictions

In [ ]:
best = "/content/runs/ar_ping_pong/ball_yolo26n_img960/weights/best.pt"
model = YOLO(best)
metrics = model.val(data=DATASET_YAML, imgsz=960)
print(metrics)

## 6. Optional Larger Comparison

Run this after the baseline works. It may need a better Colab GPU.

In [ ]:
# from ultralytics import YOLO
# model = YOLO("yolo26s.pt")
# results = model.train(
#     data=DATASET_YAML,
#     epochs=100,
#     imgsz=960,
#     batch=-1,
#     patience=25,
#     project="/content/runs/ar_ping_pong",
#     name="ball_yolo26s_img960",
# )